In [ ]:
import numpy as np
import os
from tqdm import tqdm
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Conv2D, BatchNormalization, Activation, Add, MaxPooling2D, GlobalAveragePooling2D, Dense, Dropout, Resizing
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from tensorflow.keras.utils import to_categorical
from collections import Counter
import matplotlib.pyplot as plt

# =====================================================================
# CARGAR Y PREPARAR DATOS
# =====================================================================
npz_folder = 'Insecta/NPZ/'

X_total = []
y_total = []

for file in tqdm(os.listdir(npz_folder)):
    if file.endswith('.npz'):
        path = os.path.join(npz_folder, file)
        try:
            data = np.load(path)
            spec = data['spec']
            label = data['label'].item() 

            if spec.shape == (1025, 313):
                X_total.append(spec)
                y_total.append(label)
            else:
                print(f"Forma inválida en: {file} -> {spec.shape}")
        except Exception as e:
            print(f"Error leyendo {file}: {e}")

print(f"\nEspectrogramas válidos cargados: {len(X_total)}")

# Convertir a arrays
X = np.array(X_total)
y = np.array(y_total)

# Filtrar clases con pocas muestras
label_counts = Counter(y)
clases_validas = [label for label, count in label_counts.items() if count >= 2]
filtro = [label in clases_validas for label in y]

X = X[filtro]
y = y[filtro]

print(f"Clases restantes: {len(set(y))}")
print(f"Muestras totales: {len(y)}")

# Codificar etiquetas
le = LabelEncoder()
y_encoded = le.fit_transform(y)
y_cat = to_categorical(y_encoded)
num_classes = len(le.classes_)

# Calcular pesos de clases
class_weights = compute_class_weight(
    'balanced', 
    classes=np.unique(y_encoded), 
    y=y_encoded
)
class_weights_dict = dict(enumerate(class_weights))
print("Pesos de clases:", class_weights_dict)

# Normalizar
X = X / np.max(X)

# Añadir dimensión de canal
X = X[..., np.newaxis]

# Dividir datos
X_train, X_val, y_train, y_val = train_test_split(
    X, y_cat, test_size=0.2, stratify=y_encoded, random_state=42
)


In [ ]:

# =====================================================================
# ARQUITECTURA RESNET PEQUEÑA
# =====================================================================
def residual_block(x, filters, kernel_size=3, stride=1, use_shortcut=False):
    """Bloque residual básico"""
    # Camino principal
    shortcut = x
    
    # Primer convolución
    x = Conv2D(filters, kernel_size, strides=stride, padding='same')(x)
    x = BatchNormalization()(x)
    x = Activation('relu')(x)
    
    # Segunda convolución
    x = Conv2D(filters, kernel_size, padding='same')(x)
    x = BatchNormalization()(x)
    
    # Camino corto si es necesario
    if use_shortcut or stride > 1:
        shortcut = Conv2D(filters, 1, strides=stride, padding='same')(shortcut)
        shortcut = BatchNormalization()(shortcut)
    
    # Fusionar caminos
    x = Add()([x, shortcut])
    x = Activation('relu')(x)
    return x

def build_resnet(input_shape, num_classes):
    """Construye ResNet pequeña adaptada a espectrogramas"""
    inputs = Input(shape=input_shape)
    
    # Reducción dimensional inicial (opcional pero recomendado)
    x = Resizing(256, 256, interpolation='bilinear')(inputs)  # Reducimos a 256x256
    
    # Capa inicial
    x = Conv2D(32, 7, strides=2, padding='same')(x)
    x = BatchNormalization()(x)
    x = Activation('relu')(x)
    x = MaxPooling2D(3, strides=2, padding='same')(x)
    
    # Bloques residuales
    x = residual_block(x, filters=32, stride=1)  # Bloque 1
    x = residual_block(x, filters=32, stride=1)  # Bloque 2
    
    x = residual_block(x, filters=64, stride=2, use_shortcut=True)  # Bloque 3 (reducción)
    x = residual_block(x, filters=64, stride=1)  # Bloque 4
    
    x = residual_block(x, filters=128, stride=2, use_shortcut=True)  # Bloque 5 (reducción)
    x = residual_block(x, filters=128, stride=1)  # Bloque 6
    
    # Capa final
    x = GlobalAveragePooling2D()(x)
    x = Dense(128, activation='relu')(x)
    outputs = Dense(num_classes, activation='softmax')(x)
    
    return Model(inputs, outputs)

# Construir modelo
input_shape = (1025, 313, 1)
model = build_resnet(input_shape, num_classes)
model.compile(
    optimizer=Adam(learning_rate=0.001),
    loss='categorical_crossentropy',
    metrics=['accuracy', 'Precision', 'Recall']
)
model.summary()


In [ ]:

# =====================================================================
# ENTRENAMIENTO
# =====================================================================
callbacks = [
    EarlyStopping(
        monitor='val_loss',
        patience=15,
        restore_best_weights=True,
        verbose=1
    ),
    ModelCheckpoint(
        'best_resnet_model.h5',
        save_best_only=True,
        monitor='val_accuracy',
        mode='max'
    )
]

batch_size = 32
epochs = 100

history = model.fit(
    X_train, y_train,
    batch_size=batch_size,
    epochs=epochs,
    validation_data=(X_val, y_val),
    class_weight=class_weights_dict,
    callbacks=callbacks,
    verbose=1
)


In [ ]:

# =====================================================================
# EVALUACIÓN Y VISUALIZACIÓN
# =====================================================================
# Gráficas de entrenamiento
plt.figure(figsize=(15, 5))
plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label='Train Accuracy')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
plt.title('Accuracy')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='Train Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.title('Loss')
plt.legend()
plt.savefig('resnet_training.png')
plt.show()

# Evaluación final
val_loss, val_acc, val_precision, val_recall = model.evaluate(X_val, y_val)
print(f"\nResultados finales:")
print(f" - Accuracy: {val_acc:.4f}")
print(f" - Precision: {val_precision:.4f}")
print(f" - Recall: {val_recall:.4f}")

# Reporte de clasificación
from sklearn.metrics import classification_report

y_pred = model.predict(X_val)
y_pred_classes = np.argmax(y_pred, axis=1)
y_true = np.argmax(y_val, axis=1)

print("\nReporte de Clasificación:")
print(classification_report(
    y_true, 
    y_pred_classes, 
    target_names=le.classes_,
    zero_division=0
))

# Matriz de confusión
from sklearn.metrics import confusion_matrix
import seaborn as sns

cm = confusion_matrix(y_true, y_pred_classes)
plt.figure(figsize=(12, 10))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=le.classes_, 
            yticklabels=le.classes_)
plt.xlabel('Predicho')
plt.ylabel('Verdadero')
plt.title('Matriz de Confusión')
plt.savefig('resnet_confusion_matrix.png')
plt.show()